# Temporal risk ranking for hypotension (simulation)

Compare gradient boosting on a snapshot of six latest vitals with a model that adds each vital's minimum, mean, variation, slope and observation count. Hold out 25% of stays per seed and evaluate AUROC, average precision and recall in the top 10% of risk scores. This transfers the time-bounded EHR risk-ranking idea from Wiens's CDI study to a different outcome. Generate simulated vital signs from latent pressure levels and slopes. Predict a later low arterial pressure observation.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'temporal-hypotension-ranking-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.model_selection import train_test_split\n\nVITALS = ("SysABP", "NISysABP", "MAP", "HR", "RespRate", "SpO2")\n\n\ndef summarize(early):\n    latest = []\n    summary = []\n    for name in VITALS:\n        values = sorted(early.get(name, []))\n        if values:\n            t, z = np.array(values).T\n            latest.append(z[-1])\n            summary.extend(\n                [\n                    z.min(),\n                    z.mean(),\n                    z.std(),\n                    (z[-1] - z[0]) / max(t[-1] - t[0], 0.25),\n                    len(z),\n                ]\n            )\n        else:\n            latest.append(np.nan)\n            summary.extend([np.nan] * 4 + [0])\n    return latest + summary\n\n\ndef run(seed):\n    x, y = dataset(seed)\n    train, test = train_test_split(\n        np.arange(len(y)), test_size=0.25, stratify=y, random_state=seed\n    )\n    results = {}\n    for method, cols in [("snapshot", len(VITALS)), ("temporal", x.shape[1])]:\n        fit = HistGradientBoostingClassifier(\n            max_iter=100,\n            max_leaf_nodes=12,\n            min_samples_leaf=24,\n            l2_regularization=3,\n            random_state=seed,\n        )\n        fit.fit(x[train, :cols], y[train])\n        p = fit.predict_proba(x[test, :cols])[:, 1]\n        order = np.argsort(-p)[: len(test) // 10]\n        results[method] = {\n            **scores(y[test], p),\n            "recall_top_10pct": float(y[test][order].sum() / y[test].sum()),\n            "eligible_stays": len(y),\n            "events": int(y.sum()),\n        }\n        save(fit, f"{method}-{seed}")\n    return results\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n    x = []\n    labels = []\n    for _ in range(3300):\n        level = rng.normal(120, 10)\n        drift = rng.normal(-0.7, 2.4)\n        block = {}\n        for i, k in enumerate(VITALS):\n            t = np.sort(rng.uniform(0, 6, rng.integers(2, 14)))\n            value = level - 7 * i + drift * t + rng.normal(0, 6, len(t))\n            block[k] = list(zip(t, value))\n        if any(y < 90 for k in VITALS[:2] for _, y in block[k]):\n            continue\n        x.append(summarize(block))\n        labels.append(int(level + 10 * drift + rng.normal(0, 8) < 90))\n    return np.asarray(x), np.asarray(labels)\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())